# QRShield v2 — 1/4: Feature Extraction

**Goal.** Evaluate whether a visual QR tamper detector generalises to tamper families it has never seen, using the QRShield v2 dataset (66,083 images: 60,000 main set + 6,083 camera-degraded benchmark; 5 tamper families; group split by base QR).

This notebook extracts two frozen feature sets once, so that the classifiers in notebook 2 train in minutes:

| Features | Description |
|---|---|
| `freq` (81-d) | Log-magnitude FFT of the grayscale image: radial profile, angular profile (low/high band), high-frequency energy ratio |
| `dino` (4 × 384-d) | Frozen DINOv2-small: `[CLS | mean | max | std]` of patch tokens (max/std pooling can retain local tamper signals that the mean dilutes) |

**Memory and session safety.** PyTorch is used here, TensorFlow is *not* imported (no GPU/RAM clash). Features are written in chunks of 4,096 images (atomic writes) so a disconnected session resumes from the last chunk.

**Notebook order**
1. `1_Feature_Extraction` (GPU): frequency and DINOv2 features, cached on Drive.
2. `2_Feature_Models_and_LOFO` (CPU is enough): `freq` and `dino` classifiers, final models and leave-one-family-out (LOFO).
3. `3_EfficientNet_Baselines_and_LOFO` (GPU): `v1`, `aug` and `v2` EfficientNetB0 models, final and LOFO.
4. `4_Adversarial_and_Final_Tables` (GPU): FGSM/PGD, all result tables and figures.

Notebooks 2 and 3 are independent of each other. Every notebook is **resumable**: finished work is stored on Drive and skipped on re-run.

**Default is the FULL study (`SMOKE = False`).** For an optional quick pipeline check, set `SMOKE = True` in the first code cell (small slice, separate `v2_eval_smoke` folders); never mix the two modes.

In [ ]:
# ── Common setup (identical in all four notebooks) ───────────────────────────
import os, gc, sys, json, time, glob, shutil, subprocess, warnings, zipfile
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running on Colab

SMOKE = False         # False = FULL study (default). True = quick pipeline check on a small slice (separate output folders).
SMOKE_ROWS = 3000     # number of manifest rows used in smoke mode
SEEDS = [0, 1, 2]
IMG = 224
FAMILIES = ['sticker_overlay', 'module_corruption', 'patch_overlay', 'logo_manipulation', 'occlusion_warp']
V1_FAMILIES = ['module_corruption', 'patch_overlay']        # the two tamper families v1 was trained on
METHODS = ['v1', 'aug', 'v2', 'freq', 'dino']

ROOT = Path(os.environ.get('QR_ROOT', '/content/drive/MyDrive/qrshield_v2'))
D = Path(os.environ.get('QR_DATA', '/content/qrshield_v2_data'))          # local copy of the dataset
ZIPS = Path(os.environ.get('QR_ZIPS', '/content/zips'))
OUT = ROOT / ('v2_eval_smoke' if SMOKE else 'v2_eval')
FEA, RES, MOD, TAB = (OUT / n for n in ('feats', 'results', 'models', 'tables'))
for p in (FEA, RES, MOD, TAB):
    p.mkdir(parents=True, exist_ok=True)
print('MODE:', 'SMOKE (small slice, NOT the real study)' if SMOKE else 'FULL study', f'| outputs -> {OUT}')

Mounted at /content/drive
MODE: FULL study | outputs -> /content/drive/MyDrive/qrshield_v2/v2_eval


In [ ]:
# ── Data: copy shards to the local disk, extract, merge the manifest ─────────
def prepare_data():
    if (D / '.done').exists():
        return
    D.mkdir(parents=True, exist_ok=True)
    ZIPS.mkdir(parents=True, exist_ok=True)
    src = sorted(glob.glob(str(ROOT / 'shard500' / 'shard_*.zip')))
    assert src, f'No shard_*.zip found in {ROOT / "shard500"}'
    for z in src:                                   # bulk copy Drive -> local disk (fast)
        dst = ZIPS / Path(z).name
        if not dst.exists():
            shutil.copy(z, dst)
    zips = sorted(glob.glob(str(ZIPS / 'shard_*.zip')))
    def unz(z):
        subprocess.run(['unzip', '-q', '-o', z, '-x', 'manifest.csv', '-d', str(D)], check=True)
    with ThreadPoolExecutor(2) as ex:
        list(ex.map(unz, zips))
    parts = []
    for z in zips:
        with zipfile.ZipFile(z) as zz:
            parts.append(pd.read_csv(zz.open('manifest.csv'), keep_default_na=False))
    merged = pd.concat(parts, ignore_index=True)
    merged.to_csv(D / 'manifest.csv', index=False)
    shutil.rmtree(ZIPS, ignore_errors=True)         # free local disk
    (D / '.done').write_text('ok')
    print('shards:', len(zips), '| manifest rows:', len(merged))

prepare_data()

man = pd.read_csv(D / 'manifest.csv', keep_default_na=False)
if SMOKE:
    man = man.iloc[:SMOKE_ROWS].copy()
need = ['image', 'set', 'split', 'base_id', 'cls', 'family', 'camera']
miss = [c for c in need if c not in man.columns]
assert not miss, f'manifest is missing columns {miss}; found {man.columns.tolist()}'

man['fp'] = str(D) + '/' + man['image'].astype(str)
man['cls_'] = man['cls'].astype(str)
man['label'] = (~man.cls_.str.startswith('benign')).astype(int)
man['fam'] = np.where(man.label == 1, man.cls_.str.replace('^tamper_', '', regex=True), 'benign')
man['base'] = man['base_id'].astype(str)
man['split_'] = man['split'].astype(str)
man['set_'] = man['set'].astype(str)
man['has_cam'] = man['camera'].astype(str).str.strip().str.lower().eq('true')
man['fidx'] = np.arange(len(man))

missing = [p for p in man.fp if not os.path.exists(p)]
assert not missing, f'{len(missing)} image files missing, e.g. {missing[:3]}'
assert set(man.fam.unique()) - {'benign'} == set(FAMILIES), man.fam.unique()

main = man[man.set_ == 'main']
tr = main[main.split_ == 'train']
va = main[main.split_ == 'val']
te = main[main.split_ == 'test']
cam = man[man.set_ == 'camera_bench']
# integrity checks: group split (no base QR in two splits); benchmark only from test bases
assert not set(tr.base) & set(te.base) and not set(tr.base) & set(va.base) and not set(va.base) & set(te.base)
assert not set(cam.base) & set(tr.base) and not set(cam.base) & set(va.base)
share = main.groupby('label').has_cam.mean()
assert 0.3 < share.min() and share.max() < 0.5, f'camera-layer share should be ~0.4 for both labels: {share.to_dict()}'
print(f'rows={len(man)} | main={len(main)} (train {len(tr)} / val {len(va)} / test {len(te)}) | camera_bench={len(cam)}')
print('camera-layer share by label:', share.round(3).to_dict())
if not SMOKE:   # safeguard: the full study must see the whole dataset
    assert len(man) > 60000 and len(tr) > 35000, f'FULL mode expects ~66,000 rows / ~42,000 train images, got {len(man)} / {len(tr)}. Check that all 24 shards are in {ROOT / "shard500"}.'

shards: 24 | manifest rows: 66083
rows=66083 | main=60000 (train 41958 / val 5810 / test 12232) | camera_bench=6083
camera-layer share by label: {0: 0.404, 1: 0.401}


## Frequency-domain features

In [ ]:
# ── Frequency-domain features (81-d) ─────────────────────────────────────────
from PIL import Image

_yy, _xx = np.mgrid[:IMG, :IMG]
_r = np.hypot(_yy - IMG // 2, _xx - IMG // 2)
_theta = np.mod(np.arctan2(_yy - IMG // 2, _xx - IMG // 2), np.pi)
NB = 56
RBIN = np.minimum((_r / 2).astype(int), NB - 1).ravel()
RCNT = np.maximum(np.bincount(RBIN, minlength=NB), 1)
ABIN = (_theta / np.pi * 12).astype(int).clip(0, 11).ravel()
HI = (_r >= 56).ravel()
LO = ((_r >= 8) & (_r < 40)).ravel()
WIN = np.outer(np.hanning(IMG), np.hanning(IMG)).astype(np.float32)

def freq_feat(path):
    """log-magnitude FFT -> radial profile (56) + angular profile low/high band (12+12) + HF energy ratio (1)."""
    a = np.asarray(Image.open(path).convert('L').resize((IMG, IMG), Image.BILINEAR), np.float32) / 255.
    a = (a - a.mean()) * WIN
    mag = np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(a)))).ravel()
    prof = np.bincount(RBIN, mag, minlength=NB) / RCNT
    ah = np.bincount(ABIN[HI], mag[HI], minlength=12) / np.maximum(np.bincount(ABIN[HI], minlength=12), 1)
    al = np.bincount(ABIN[LO], mag[LO], minlength=12) / np.maximum(np.bincount(ABIN[LO], minlength=12), 1)
    pw = np.expm1(mag) ** 2
    return np.concatenate([prof, ah, al, [pw[HI].sum() / (pw.sum() + 1e-9)]]).astype(np.float32)

def extract_freq(paths):
    with ThreadPoolExecutor(os.cpu_count() or 2) as ex:
        return np.stack(list(ex.map(freq_feat, paths)))

## DINOv2 features

In [ ]:
# ── DINOv2 features (frozen backbone) ────────────────────────────────────────
DINO_MODEL = 'facebook/dinov2-small'     # 384-d tokens. Larger option: 'facebook/dinov2-base' (change DINO_TAG too)
DINO_TAG = 'dino_small'

def make_dino_extractor(model_name=DINO_MODEL, batch_size=128, workers=2):
    """Per image: [CLS | mean(patch tokens) | max(patch tokens) | std(patch tokens)]  ->  4 x hidden_dim.
    Tampering is local, so max/std pooling can keep signals that the mean alone dilutes."""
    import torch
    from transformers import AutoModel
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = AutoModel.from_pretrained(model_name).to(dev).eval()
    if dev == 'cuda':
        model = model.half()
    mean = np.array([0.485, 0.456, 0.406], np.float32)
    std = np.array([0.229, 0.224, 0.225], np.float32)

    class Imgs(torch.utils.data.Dataset):
        def __init__(self, paths): self.paths = paths
        def __len__(self): return len(self.paths)
        def __getitem__(self, i):
            a = np.asarray(Image.open(self.paths[i]).convert('RGB').resize((IMG, IMG), Image.BICUBIC), np.float32) / 255.
            return torch.from_numpy(((a - mean) / std).transpose(2, 0, 1).copy())

    def extract(paths):
        outs = []
        loader = torch.utils.data.DataLoader(Imgs(paths), batch_size=batch_size, num_workers=workers)
        with torch.inference_mode():
            for x in loader:
                x = x.to(dev)
                x = x.half() if dev == 'cuda' else x
                h = model(pixel_values=x).last_hidden_state.float()
                p = h[:, 1:]
                outs.append(torch.cat([h[:, 0], p.mean(1), p.amax(1), p.std(1)], 1).cpu().numpy())
        return np.concatenate(outs)
    return extract

## Extraction (chunked, resumable)
The first cell below extracts frequency features (CPU); the second extracts DINOv2 features (GPU). Re-run a cell after a disconnect to continue from the last finished chunk. Extraction time has not been measured; the chunk log shows the speed.

In [ ]:
# ── Chunked, resumable feature store on Drive ─────────────────────────────────
CHUNK = 4096

def extract_chunked(tag, paths, fn, dtype):
    d = FEA / tag
    d.mkdir(parents=True, exist_ok=True)
    meta = dict(n=len(paths), first=paths[0], last=paths[-1])
    mp = d / 'meta.json'
    if mp.exists():
        assert json.load(open(mp)) == meta, f'Manifest changed since {d} was written; delete that folder.'
    else:
        json.dump(meta, open(mp, 'w'))
    n_chunks = (len(paths) + CHUNK - 1) // CHUNK
    for i in range(n_chunks):
        f = d / f'part_{i:04d}.npy'
        if f.exists():
            continue                                  # resume: finished chunks are skipped
        t0 = time.time()
        arr = fn(paths[i * CHUNK:(i + 1) * CHUNK]).astype(dtype)
        tmp = d / f'part_{i:04d}.tmp.npy'
        np.save(tmp, arr)
        os.replace(tmp, f)                            # atomic
        print(f'{tag}: chunk {i + 1}/{n_chunks} done ({time.time() - t0:.0f}s)', flush=True)
    return n_chunks

def load_feats(tag):
    d = FEA / tag
    parts = [np.load(f) for f in sorted(d.glob('part_*.npy')) if '.tmp' not in f.name]
    return np.concatenate(parts)

paths = list(man.fp)
t0 = time.time()
extract_chunked('freq', paths, extract_freq, np.float32)
print(f'frequency features finished in {(time.time() - t0) / 60:.1f} min')

freq: chunk 1/17 done (17s)
freq: chunk 2/17 done (18s)
freq: chunk 3/17 done (18s)
freq: chunk 4/17 done (17s)
freq: chunk 5/17 done (17s)
freq: chunk 6/17 done (17s)
freq: chunk 7/17 done (17s)
freq: chunk 8/17 done (16s)
freq: chunk 9/17 done (17s)
freq: chunk 10/17 done (17s)
freq: chunk 11/17 done (17s)
freq: chunk 12/17 done (17s)
freq: chunk 13/17 done (17s)
freq: chunk 14/17 done (17s)
freq: chunk 15/17 done (16s)
freq: chunk 16/17 done (16s)
freq: chunk 17/17 done (2s)
frequency features finished in 4.5 min


In [ ]:
t0 = time.time()
dino_fn = make_dino_extractor()
extract_chunked(DINO_TAG, paths, dino_fn, np.float16)
print(f'DINOv2 features finished in {(time.time() - t0) / 60:.1f} min')

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 88.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

dino_small: chunk 1/17 done (16s)
dino_small: chunk 2/17 done (14s)
dino_small: chunk 3/17 done (13s)
dino_small: chunk 4/17 done (15s)
dino_small: chunk 5/17 done (15s)
dino_small: chunk 6/17 done (15s)
dino_small: chunk 7/17 done (15s)
dino_small: chunk 8/17 done (15s)
dino_small: chunk 9/17 done (17s)
dino_small: chunk 10/17 done (15s)
dino_small: chunk 11/17 done (16s)
dino_small: chunk 12/17 done (16s)
dino_small: chunk 13/17 done (16s)
dino_small: chunk 14/17 done (15s)
dino_small: chunk 15/17 done (16s)
dino_small: chunk 16/17 done (15s)
dino_small: chunk 17/17 done (2s)
DINOv2 features finished in 4.5 min


In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────
Xf, Xd = load_feats('freq'), load_feats(DINO_TAG)
print('freq:', Xf.shape, '| dino:', Xd.shape, '| manifest rows:', len(man))
assert len(Xf) == len(Xd) == len(man), 'feature rows do not match the manifest'
assert np.isfinite(Xf).all() and np.isfinite(Xd.astype(np.float32)).all(), 'NaN/Inf in features'
print('OK. Next: notebook 2 (feature models + LOFO). Notebook 3 (EfficientNet) is independent and can run in parallel sessions.')

freq: (66083, 81) | dino: (66083, 1536) | manifest rows: 66083
OK. Next: notebook 2 (feature models + LOFO). Notebook 3 (EfficientNet) is independent and can run in parallel sessions.
